# 04. Variable Window: Shortest Window and Counting Subarrays

Two more variable-window styles:
- **Shortest** window that satisfies a condition.
- **Counting** all subarrays that satisfy a condition.

**Learning order:** SHORTEST TEMPLATE → MIN SIZE SUBARRAY SUM → MINIMUM WINDOW SUBSTRING → COUNTING IDEA → PRODUCT LESS THAN K → EXACTLY K TRICK → INTERVIEW THINKING

## 1. Shortest window: the template flips

For *longest*, we shrink while **invalid**, and record after.
For *shortest*, we shrink while **valid**, and record **inside** the loop.

```python
left = 0
for right in range(len(arr)):
    add(arr[right])
    while window_is_valid():
        best = min(best, right - left + 1)
        remove(arr[left])
        left += 1
```

Reason: once valid, we try to make it as small as possible. Real life: "smallest number of days that give at least ₹10,000 in sales".

## 2. Minimum Size Subarray Sum

Smallest length subarray with sum ≥ `target`. All numbers are **positive**.

**Positive numbers matter:** adding an element always increases the sum, removing always decreases it. That gives a clear rule to expand or shrink.

In [1]:
def min_subarray_len(target, nums):
    left = 0
    total = 0
    best = float("inf")
    for right in range(len(nums)):
        total += nums[right]
        while total >= target:
            best = min(best, right - left + 1)
            total -= nums[left]
            left += 1
    return 0 if best == float("inf") else best

print(min_subarray_len(7, [2, 3, 1, 2, 4, 3]))   # 2  ([4, 3])

2


## 3. Minimum Window Substring (Hard, but a pattern)

Smallest substring of `s` that contains **all characters of `t`** (with counts).

**Plan:**
- `need[ch]` = how many of `ch` we still need. `missing` = total characters still missing.
- Entering char: if `need[ch] > 0` it helps → `missing -= 1`. Then `need[ch] -= 1` (can go negative = extra copies).
- When `missing == 0` the window is valid → shrink from the left, recording the best.
- Leaving char: `need[ch] += 1`. If it becomes `> 0`, we lost something we needed → `missing += 1`.

In [2]:
from collections import Counter

def min_window(s, t):
    if not t or len(t) > len(s):
        return ""
    need = Counter(t)
    missing = len(t)
    left = 0
    best_start, best_len = 0, float("inf")
    for right, ch in enumerate(s):
        if need[ch] > 0:
            missing -= 1
        need[ch] -= 1
        while missing == 0:                      # window is valid
            if right - left + 1 < best_len:
                best_start, best_len = left, right - left + 1
            need[s[left]] += 1
            if need[s[left]] > 0:                # lost a needed char
                missing += 1
            left += 1
    return "" if best_len == float("inf") else s[best_start:best_start + best_len]

print(min_window("ADOBECODEBANC", "ABC"))   # BANC
print(repr(min_window("a", "aa")))          # ''

BANC
''


## 4. Counting subarrays

Key fact: if window `[left..right]` is valid for a "**at most**" style condition, then **every subarray ending at `right` and starting from `left` to `right` is also valid**. That is `right - left + 1` subarrays.

So after shrinking to make the window valid, add `right - left + 1` to the count.

Real life: "how many ways can I pick consecutive days with total sales less than a limit?"

In [3]:
def num_subarray_product_less_than_k(nums, k):
    if k <= 1:
        return 0
    product = 1
    left = 0
    count = 0
    for right in range(len(nums)):
        product *= nums[right]
        while product >= k:
            product //= nums[left]
            left += 1
        count += right - left + 1       # all subarrays ending at right
    return count

print(num_subarray_product_less_than_k([10, 5, 2, 6], 100))   # 8

8


## 5. "Exactly K" = "at most K" − "at most K−1"

Counting subarrays with **exactly** `k` of something is hard directly, because the window is not monotonic. But "at most k" is easy, so:

`exactly(k) = at_most(k) - at_most(k - 1)`

Example: *Count Number of Nice Subarrays* = subarrays with exactly `k` odd numbers.

In [4]:
def number_of_nice_subarrays(nums, k):
    def at_most(k):
        if k < 0:
            return 0
        left = 0
        odd = 0
        count = 0
        for right in range(len(nums)):
            odd += nums[right] % 2
            while odd > k:
                odd -= nums[left] % 2
                left += 1
            count += right - left + 1
        return count

    return at_most(k) - at_most(k - 1)

print(number_of_nice_subarrays([1, 1, 2, 1, 1], 3))                  # 2
print(number_of_nice_subarrays([2, 4, 6], 1))                        # 0
print(number_of_nice_subarrays([2, 2, 2, 1, 2, 2, 1, 2, 2, 2], 2))   # 16

2
0
16


## 6. Interview thinking

**Signals:**
- "smallest / minimum length subarray ..." → shortest template
- "number of subarrays with ..." → counting `right - left + 1`
- "exactly k" → `atMost(k) - atMost(k-1)`

**Common mistakes:** recording the shortest answer *outside* the `while`, using the shortest template with negative numbers, forgetting the `k <= 1` guard in product problems, counting only `1` per step instead of `right - left + 1`, wrong `missing` bookkeeping in Minimum Window.

## Quick Revision

- Longest: shrink while invalid. Shortest: shrink while **valid**, record inside the loop.
- Min window substring: `need` counter + `missing` counter.
- Count subarrays ending at `right`: `right - left + 1`.
- Exactly k = at most k − at most (k−1).
- Sum/product based shrinking needs **positive** numbers.